Problem Statement

The objective of this project is to develop a deep learning system that can understand and classify textual data based on its sentiment. We will first use the IMDB movie review dataset as a controlled learning problem and progressively build and compare Simple RNN, LSTM, and GRU models using PyTorch.

The project will begin with text preprocessing, tokenization, sequence padding, and word embeddings. We will then train recurrent neural network models to classify movie reviews as positive or negative. The models will be evaluated and compared using appropriate performance metrics, training behavior, and error analysis.

After establishing a strong understanding of sequence modeling, the learned techniques will be applied to a more practical YouTube Comment Analyzer, where comments can be classified according to meaningful categories such as sentiment or toxicity, depending on the available dataset and labeling quality.

IMDB movie reviews = training/learning environment to understand RNNs, LSTMs, and GRUs.

Then we take what we learned and apply it to the real-world YouTube Comment Analyzer.

In [18]:
import random 
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
import torch
print("PyTorch version:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())#torch.cuda.is_available() → checks whether a GPU is available.

PyTorch version: 2.14.0+cpu
CUDA available: False


The GPU check matters because training neural networks can be much faster on a GPU.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
!pip install torch datasets

PyTorch is the deep-learning framework we use to build/train the RNN, while Hugging Face Datasets is a separate library we use to load the IMDB data.

you do not need to install huggingface_hub separately for what we are doing
You already ran:
pip install torch datasets

The datasets package uses huggingface_hub internally, so it is normally installed as a dependency.

In [5]:
import torch
from datasets import load_dataset

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu


In [6]:
import datasets
import huggingface_hub

print(datasets.__version__)
print(huggingface_hub.__version__)

5.0.1
1.30.0


In [8]:
#Load IMDB
dataset=load_dataset('stanfordnlp/imdb')
print(dataset)

r:\C\PYTHON PROGRAMS\Deep Learning Project 4(RNN)\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aishw\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 350855.07 examples/s

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Yes. stanfordnlp/imdb is the Hugging Face dataset identifier (repository address/name) for the IMDb dataset. It tells load_dataset() exactly which dataset repository to fetch.

In [10]:
print(dataset['train'][0])
print(dataset["train"][1])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

The IMDB dataset contains movie reviews with labels:
text → movie review
label → 0 (negative) or 1 (positive)

In [16]:
print(dataset['test'][24000])

{'text': "The movie is about a girl who's not going to a bonfire only because she's baby-sitting that night. Nothing weird about that, right? Until ... The phone rings. Until ... The phone rings again. And again ... And again. Those are not some stupid prank calls. This is for real. If you wanna see how the girl reacts, just watch the movie.<br /><br />Great atmosphere filled with scary sounds. Very well performed by young Camilla Belle who got the lead role. I see in her some great potential to become a good actress. This is more than only a decent thriller, I have no idea why it's so underrated. Anyway, on my opinion this movie deserves more than only 4/10. 24% of all voters rated the movie with 1. Get serious, people. You couldn't get a better thriller for a title like this.", 'label': 1}


Our project uses:
TRAIN
25,000 labeled reviews
        ↓
Learn

TEST
25,000 labeled reviews
        ↓
Evaluate

We first loaded the raw dataset because we needed to confirm what it contains. Now preprocessing is the next major stage.

Text preprocessing
What are we doing?

Our dataset currently contains:
"I rented I AM CURIOUS-YELLOW from my video store..."
PyTorch cannot send this English sentence directly into an RNN.

We first convert it into tokens:
"I rented this movie"
        ↓
["i", "rented", "this", "movie"]

Then later:
["i", "rented", "this", "movie"]
        ↓
[15, 428, 37, 892]

The numbers are the word IDs in our vocabulary.
Why are we doing this?

Because the RNN ultimately works with numerical tensors, not words.

So our preprocessing pipeline is:

Raw text
   ↓
Cleaning
   ↓
Tokenization
   ↓
Vocabulary
   ↓
Integer encoding
   ↓
Padding
   ↓
Tensor
   ↓
Embedding
   ↓
RNN

One important decision

For this project, I recommend that we build the tokenizer/vocabulary ourselves rather than using a ready-made NLP tokenizer.

In [21]:
text=dataset['train'][0]['text']
print(text)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [23]:
tex=dataset['train'][0]['label']
print(tex)

0


Now we will perform the first preprocessing operation: basic text cleaning.

Our reviews contain HTML tags such as:
<br /><br />
These are not meaningful words for sentiment classification, so we should remove them.

In [24]:
import re
text=dataset['train'][0]['text']
clean_text=re.sub(r"<br\s*/?>",' ',text)
print(clean_text)

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.  The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.  What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot

Tokenization
We want to convert:
"I rented I AM CURIOUS-YELLOW from my video store"
into individual tokens:
["i", "rented", "i", "am", "curious-yellow", "from", "my", "video", "store"]

For our first implementation, we'll use a simple Python tokenizer so you understand what is happening rather than hiding it inside a library.

In [27]:
def tokenize(text):
    return text.lower().split()

tokens=tokenize(clean_text)
print(tokens[:20])

['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


Why .lower()?
These:
Movie
movie
MOVIE
should normally be treated as the same word.
So we convert everything to lowercase.

Why .split()?
.split() separates the text wherever there is whitespace:

"I love this movie"
        ↓
["i", "love", "this", "movie"]

This is our first simple tokenizer.

Later, when we build the vocabulary, we'll convert these tokens into integer IDs.

So the pipeline currently is:
Raw review
    ↓
Remove HTML
    ↓
Lowercase
    ↓
Split into tokens
    ↓
["i", "rented", "i", "am", ...]

We need:

25,000 raw reviews
        ↓
tokenize() on every review
        ↓
25,000 lists of tokens

In [28]:
# So first let's apply our tokenizer to the whole training set.
train_tokens=[tokenize(text) for text in dataset['train']['text']]
print("Number of reviews:",len(train_tokens))
print("First review tokens:",train_tokens[0][:20])

Number of reviews: 25000
First review tokens: ['i', 'rented', 'i', 'am', 'curious-yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was']


train_tokens[0][:20] means:
train_tokens[0] → first review's complete token list
[:20] → take only the first 20 tokens

In [ ]:
#we’ll build the vocabulary from the tokenized training reviews only
#This is the step where every word gets its consistent integer ID.




Now comes the important part: Vocabulary
We cannot assign random IDs separately for every review. We need one common vocabulary for the entire training dataset.
For example:
<padded> → 0
<unk>    → 1
the      → 2
movie    → 3
good     → 4
bad      → 5
...
Then every review uses the same mapping.

Vocabulary = one common dictionary that assigns a unique number (ID) to each word.
So if: "the" → 2
then every occurrence of "the" in every review will be represented by 2.


What are <PAD> and <UNK>?
<PAD> = padding token. We use it to fill shorter reviews so all sequences have the same length.

<UNK> = unknown token. Used when a word is not present in our vocabulary (for example, a rare word that we decided not to include).

The important point is:<UNK> does not mean “gibberish.” It means “a valid word that our vocabulary does not contain.”

For example, suppose we keep only the 10,000 most frequent words:
the       → 2
movie     → 3
good      → 4

If "fantabulous" exists in English but is too rare and therefore was not included in our 10,000-word vocabulary, then:
fantabulous → <UNK> → 1
So <UNK> means unknown to our vocabulary, not unknown to the English language.